# 03 · Two-view geometry and the first point cloud

**Goal:** follow the initial pair from verified matches to a coloured sparse point cloud, and check every step: epipolar geometry, relative pose, triangulation quality and — when the dataset has ground truth — accuracy.

Set `SCENE_DIR` (or `MV3D_SCENE`). A `camera.yaml` in the scene folder is used as calibration; a `gt_cameras.json` enables the ground-truth section (see `scripts/download_middlebury.py`).

In [ ]:
%matplotlib inline
import os
import tempfile
from pathlib import Path

import matplotlib.pyplot as plt
import numpy as np
from IPython.display import Image, display

from src.config import load_config
from src.pipeline import run_pipeline
from src.pointcloud.coloring import color_points
from src.visualization.cameras import plot_two_view
from src.visualization.epipolar import draw_epipolar_lines
from src.visualization.matches import draw_matches
from src.visualization.reprojection import plot_reprojection_histogram
from src.visualization.viewer import RenderUnavailable, render_scene

SCENE_DIR = Path(os.environ.get('MV3D_SCENE', '../data/raw/templeRing'))
CONFIG_PATH = None  # None -> configs/default.yaml

# Notebooks run from notebooks/: keep caches in the repository's outputs/cache.
overrides = {'output.cache_dir': '../outputs/cache'}
if (SCENE_DIR / 'camera.yaml').exists():
    overrides['camera.calibration_file'] = str(SCENE_DIR / 'camera.yaml')
config = load_config(CONFIG_PATH).with_overrides(overrides)
SCENE_DIR.resolve()

## 1. Run the pipeline up to triangulation

In [ ]:
r = run_pipeline(SCENE_DIR, config, until='triangulate')
pair, s = r.initial_pair, r.structure
names = r.names
assert pair is not None, 'no initial pair: capture views with more baseline'
a, b = r.images[pair.i], r.images[pair.j]
print(f'{len(names)} images, {sum(g.success for g in r.geometries)} verified pairs')
print(f'initial pair: {a.name} - {b.name}; {pair.geometry.n_inliers} inliers, '
      f'model {pair.geometry.model}, parallax {pair.pose.median_parallax_deg:.1f} deg, '
      f'rotation {pair.pose.rotation_deg:.1f} deg, H/F {pair.geometry.homography_ratio:.2f}')

## 2. Verified matches and epipolar lines

Left: inliers (green) and rejected matches (red). Right: every point must lie on the line of its own colour in the other image.

In [ ]:
k = next(n for n, m in enumerate(r.matches) if (m.i, m.j) == (pair.i, pair.j))
m, g = r.matches[k], r.geometries[k]
image_a, image_b = a.image, b.image
matches_img = draw_matches(image_a, image_b, r.features[m.i], r.features[m.j], m,
                           inlier_mask=g.inlier_mask)
epi_img = draw_epipolar_lines(image_a, image_b, pair.x_i, pair.x_j, g.F)
fig, axes = plt.subplots(2, 1, figsize=(12, 9))
panels = [(matches_img, 'matches'), (epi_img, 'epipolar lines')]
for ax, (img, title) in zip(axes, panels, strict=True):
    ax.imshow(img[..., ::-1])
    ax.set_title(title)
    ax.axis('off')
fig.tight_layout()

## 3. Triangulation quality

Counts per filter, the reprojection-error distribution and the structure seen from above and from the side (camera of image i at the origin, baseline length 1).

In [ ]:
tri = s.triangulation
print(tri.counts)
print('reprojection of valid points:', tri.reprojection)
display(plot_reprojection_histogram(tri.errors, config.triangulation.max_reprojection_error_px,
                                    title=f'{a.name} - {b.name}'))
display(plot_two_view(pair.pose.R, pair.pose.t, s.points, (a.name, b.name),
                      title=f'{len(s.points)} valid points'))

## 4. The coloured sparse point cloud

Colours are sampled (bilinearly) at every observation and averaged. The close-up is rendered offscreen with Open3D; run `python -m src.cli view <ply>` for an interactive window.

In [ ]:
colors = color_points([image_a, image_b], s.observations, len(tri.valid))[tri.valid]
sizes = [(a.width, a.height), (b.width, b.height)]
render_dir = Path(tempfile.mkdtemp())
try:
    for focus in ('points', 'cameras'):
        path = render_scene(s.points, colors, s.cameras, sizes,
                            render_dir / f'{focus}.png', width=900, height=675, focus=focus)
        display(Image(filename=str(path)))
except RenderUnavailable as exc:
    print('offscreen rendering unavailable here:', exc)

## 5. Accuracy against ground truth

Only when `gt_cameras.json` exists. Pose errors compare directions (translation has no scale). Structure is compared after the best similarity alignment, because a two-view reconstruction is only defined up to rotation, translation and scale.

In [ ]:
gt_path = SCENE_DIR / 'gt_cameras.json'
if gt_path.exists():
    from src.evaluation.ground_truth import load_cameras_json, relative_pose
    from src.evaluation.metrics import (
        rotation_error_deg,
        translation_direction_error_deg,
        two_view_structure_errors,
    )
    gt = load_cameras_json(gt_path)
    R_true, t_true = relative_pose(gt[a.name], gt[b.name])
    rot = rotation_error_deg(pair.pose.R, R_true)
    trans = translation_direction_error_deg(pair.pose.t, t_true)
    print(f'rotation error {rot:.2f} deg, translation direction error {trans:.2f} deg')
    res = two_view_structure_errors(
        s, gt[a.name], gt[b.name], scale_i=a.scale, scale_j=b.scale
    )
    print(f'3-D shape error: median {np.median(res.errors):.3%} of scene size, '
          f'90th percentile {np.percentile(res.errors, 90):.3%}; '
          f'scale vs true baseline {res.scale_ratio:.3f}')
    fig, ax = plt.subplots(figsize=(6, 3))
    ax.hist(100 * res.errors, bins=40, color='tab:green')
    ax.set_xlabel('3-D error after alignment (% of scene size)')
    ax.set_ylabel('points')
    fig.tight_layout()
else:
    print('no gt_cameras.json in', SCENE_DIR)

## 6. Findings

- Does the close-up resemble the object in the photos?
- Rejected points per filter and why:
- Reprojection error and triangulation angles:
- Accuracy against ground truth (if available):